# Credit Risk — Baseline Models

Order:
1. Split the data into **train / validation / test = 60 / 20 / 20** (TARGET rate preserved, fixed seed). **The test set is not touched until the model is final.**
2. Simple reference: the same probability for everyone (train default rate).
3. Logistic Regression (application columns).
4. LightGBM — application columns first, then all columns (application + bureau + previous_application).

All comparisons use the **validation** set.

> Data comes from `mart.features_application`, which is built by `run_pipeline.py`.

In [1]:
import os
import time

import numpy as np
import pandas as pd
import psycopg2
import lightgbm as lgb
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42

# Postgres NUMERIC -> float (otherwise pandas receives Decimal objects)
DEC2FLOAT = psycopg2.extensions.new_type(
    psycopg2.extensions.DECIMAL.values, "DEC2FLOAT", lambda v, cur: float(v) if v is not None else None
)
psycopg2.extensions.register_type(DEC2FLOAT)

load_dotenv()
engine = create_engine(
    f"postgresql+psycopg2://{os.environ['POSTGRES_USER']}:{os.environ['POSTGRES_PASSWORD']}"
    f"@localhost:{os.environ['POSTGRES_PORT']}/{os.environ['POSTGRES_DB']}"
)

## 1. Data

In [2]:
t0 = time.time()
df = pd.read_sql("SELECT * FROM mart.features_application ORDER BY sk_id_curr", engine)
df = df[[c for c in df.columns if not c.startswith("fe_")]]  # feature engineering lives in model_selection.ipynb
print(f"{df.shape[0]:,} rows, {df.shape[1]} columns — {time.time() - t0:.0f} s")
assert df["sk_id_curr"].is_unique

bureau_cols = [c for c in df.columns if c.startswith("bureau_") and c != "bureau_has_record"]
prev_cols = [c for c in df.columns if c.startswith("prev_") and c != "prev_has_record"]
app_cols_db = [c for c in df.columns if c not in bureau_cols + prev_cols + ["bureau_has_record", "prev_has_record"]]

307,511 rows, 158 columns — 14 s


In [3]:
ID, TARGET = "sk_id_curr", "target"

APP_FEATURES = [c for c in app_cols_db if c not in (ID, TARGET)]
ALL_FEATURES = APP_FEATURES + bureau_cols + ["bureau_has_record"] + prev_cols + ["prev_has_record"]

CAT_FEATURES = [c for c in ALL_FEATURES if not pd.api.types.is_numeric_dtype(df[c])]

print(f"application features: {len(APP_FEATURES)}")
print(f"all features:         {len(ALL_FEATURES)}  (categorical: {len(CAT_FEATURES)})")
print(f"default rate:         {df[TARGET].mean():.4f}")

application features: 121
all features:         156  (categorical: 16)
default rate:         0.0807


## 2. Train / validation / test split (60 / 20 / 20)

First 20% is held out as test, then 25% of the remaining 80% becomes validation (= 20% of the total). Both steps use `stratify=TARGET`. Split IDs are written to `data/split_ids.csv` so the split stays fixed.

In [4]:
train_val, test = train_test_split(df, test_size=0.20, stratify=df[TARGET], random_state=SEED)
train, val = train_test_split(train_val, test_size=0.25, stratify=train_val[TARGET], random_state=SEED)

os.makedirs("data", exist_ok=True)
pd.concat([
    train[[ID]].assign(split="train"),
    val[[ID]].assign(split="val"),
    test[[ID]].assign(split="test"),
]).sort_values(ID).to_csv("data/split_ids.csv", index=False)

summary = pd.DataFrame({
    "rows": [len(train), len(val), len(test)],
    "share": [len(train) / len(df), len(val) / len(df), len(test) / len(df)],
    "default rate": [train[TARGET].mean(), val[TARGET].mean(), test[TARGET].mean()],
}, index=["train", "val", "test"])
summary.style.format({"rows": "{:,}", "share": "{:.1%}", "default rate": "{:.4%}"})

,rows,share,default rate
train,"184,506",60.0%,8.0729%
val,"61,502",20.0%,8.0729%
test,"61,503",20.0%,8.0728%


In [5]:
# The test set is NOT used from here on until model selection is finished.
y_train, y_val = train[TARGET].to_numpy(), val[TARGET].to_numpy()

results = {}

def evaluate(name, y_true, p):
    results[name] = {
        "ROC-AUC": roc_auc_score(y_true, p),
        "Gini": 2 * roc_auc_score(y_true, p) - 1,
        "PR-AUC": average_precision_score(y_true, p),
        "LogLoss": log_loss(y_true, p),
        "Brier": brier_score_loss(y_true, p),
    }
    return pd.DataFrame(results).T.round(4)

## 3. Simple reference: the same rate for everyone

Every customer gets the train default rate. ROC-AUC is 0.5 by definition; the useful reference values are LogLoss / Brier.

In [6]:
p_const = np.full(len(val), y_train.mean())
evaluate("Constant rate", y_val, p_const)

,ROC-AUC,Gini,PR-AUC,LogLoss,Brier
Constant rate,0.5,0.0,0.0807,0.2805,0.0742


## 4. Logistic Regression (application columns)

- Numeric: median imputation + missing indicator, standardization
- Categorical: missing values become a `missing` category, one-hot
- Preprocessing is fitted on train only.

In [7]:
app_cat = [c for c in APP_FEATURES if c in CAT_FEATURES]
app_num = [c for c in APP_FEATURES if c not in CAT_FEATURES]

preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler()),
    ]), app_num),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), app_cat),
])

logreg = Pipeline([
    ("prep", preprocess),
    ("model", LogisticRegression(max_iter=2000, random_state=SEED)),
])

t0 = time.time()
logreg.fit(train[APP_FEATURES].astype({c: object for c in app_cat}), y_train)
print(f"training: {time.time() - t0:.0f} s")

p_logreg = logreg.predict_proba(val[APP_FEATURES].astype({c: object for c in app_cat}))[:, 1]
evaluate("LogReg (application)", y_val, p_logreg)

training: 4 s


,ROC-AUC,Gini,PR-AUC,LogLoss,Brier
Constant rate,0.5000,0.0000,0.0807,0.2805,0.0742
LogReg (application),0.7459,0.4919,0.2235,0.2498,0.0687


## 5. LightGBM

Categorical columns are converted to the `category` dtype (categories taken from train). LightGBM handles missing values natively, so there is no imputation. The number of trees is chosen by early stopping on validation AUC.

> Note: validation is used both for early stopping and for comparison, so the scores are slightly optimistic; the unbiased result comes from the test set.

In [8]:
def to_lgb_frame(frame, features, categories):
    X = frame[features].copy()
    for c in features:
        if c in categories:
            X[c] = pd.Categorical(X[c], categories=categories[c])
    return X

train_categories = {c: sorted(train[c].dropna().unique()) for c in CAT_FEATURES}

LGB_PARAMS = dict(
    objective="binary",
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=100,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    n_estimators=5000,
    random_state=SEED,
    verbose=-1,
)

def fit_lgb(features):
    X_tr = to_lgb_frame(train, features, train_categories)
    X_va = to_lgb_frame(val, features, train_categories)
    model = lgb.LGBMClassifier(**LGB_PARAMS)
    t0 = time.time()
    model.fit(
        X_tr, y_train,
        eval_X=(X_va,),
        eval_y=(y_val,),
        eval_metric="auc",
        callbacks=[lgb.early_stopping(100, verbose=False)],
    )
    print(f"training: {time.time() - t0:.0f} s, best iteration: {model.best_iteration_}")
    return model, model.predict_proba(X_va)[:, 1]

### 5a. LightGBM — application columns

In [9]:
lgb_app, p_lgb_app = fit_lgb(APP_FEATURES)
evaluate("LightGBM (application)", y_val, p_lgb_app)

training: 5 s, best iteration: 320


,ROC-AUC,Gini,PR-AUC,LogLoss,Brier
Constant rate,0.5000,0.0000,0.0807,0.2805,0.0742
LogReg (application),0.7459,0.4919,0.2235,0.2498,0.0687
LightGBM (application),0.7556,0.5112,0.2370,0.2472,0.0682


### 5b. LightGBM — all columns (application + bureau + previous_application)

In [10]:
lgb_all, p_lgb_all = fit_lgb(ALL_FEATURES)
evaluate("LightGBM (all columns)", y_val, p_lgb_all)

training: 4 s, best iteration: 220


,ROC-AUC,Gini,PR-AUC,LogLoss,Brier
Constant rate,0.5000,0.0000,0.0807,0.2805,0.0742
LogReg (application),0.7459,0.4919,0.2235,0.2498,0.0687
LightGBM (application),0.7556,0.5112,0.2370,0.2472,0.0682
LightGBM (all columns),0.7706,0.5412,0.2522,0.2430,0.0673


In [11]:
importance = (pd.DataFrame({
        "feature": ALL_FEATURES,
        "gain": lgb_all.booster_.feature_importance(importance_type="gain"),
    })
    .assign(source=lambda d: np.select(
        [d.feature.isin(bureau_cols + ["bureau_has_record"]), d.feature.isin(prev_cols + ["prev_has_record"])],
        ["bureau", "previous_application"], "application"))
    .sort_values("gain", ascending=False)
    .assign(gain_share=lambda d: d.gain / d.gain.sum()))

print("Total gain share by source:")
display(importance.groupby("source")["gain_share"].sum().sort_values(ascending=False).map("{:.1%}".format))
importance.head(20).reset_index(drop=True).style.format({"gain": "{:,.0f}", "gain_share": "{:.2%}"})

Total gain share by source:


source
application             80.0%
previous_application    15.7%
bureau                   4.3%
Name: gain_share, dtype: str

,feature,gain,source,gain_share
0,ext_source_3,"36,922",application,19.01%
1,ext_source_2,"31,945",application,16.44%
2,organization_type,"16,257",application,8.37%
3,ext_source_1,"11,753",application,6.05%
4,days_employed,"5,427",application,2.79%
5,prev_cc_avg_utilization,"4,475",previous_application,2.30%
6,prev_avg_credit_to_application_ratio,"4,337",previous_application,2.23%
7,days_birth,"4,309",application,2.22%
8,occupation_type,"4,278",application,2.20%
9,prev_avg_payment_ratio,"4,249",previous_application,2.19%


## 6. Comparison (validation)

In [12]:
pd.DataFrame(results).T.round(4)

,ROC-AUC,Gini,PR-AUC,LogLoss,Brier
Constant rate,0.5000,0.0000,0.0807,0.2805,0.0742
LogReg (application),0.7459,0.4919,0.2235,0.2498,0.0687
LightGBM (application),0.7556,0.5112,0.2370,0.2472,0.0682
LightGBM (all columns),0.7706,0.5412,0.2522,0.2430,0.0673


## 7. Test set

The final evaluation lives in `final_test.ipynb`, which runs once after model selection in `model_selection.ipynb`. The sketch below is kept for reference only.

```python
# X_te = to_lgb_frame(test, ALL_FEATURES, train_categories)
# p_test = final_model.predict_proba(X_te)[:, 1]
# roc_auc_score(test[TARGET], p_test)
```